# Slide Longform Video Generator
Render slide-based longform video di Google Colab (GPU accelerated)

**Repo**: https://github.com/Deady456/faceless-video-pipeline

## 1. Setup Environment

In [ ]:
# Clone pipeline repo
!git clone https://github.com/Deady456/faceless-video-pipeline.git
%cd faceless-video-pipeline

# Install dependencies
!pip install -q moviepy playwright edge-tts
!playwright install chromium

# Install ffmpeg
!apt-get -qq install -y ffmpeg

print('\n✅ Setup selesai')

## 2. Buat Slide Plan (plan.json)
Edit array di bawah. Setiap object = 1 slide.

**Tipe slide**: `title`, `statement`, `data`, `concept`, `comparison`, `tags`, `list`, `handwritten`, `transition`, `end`

In [ ]:
# === EDIT SLIDE PLAN DI SINI ===
import json, os, datetime

slides = [
  {"type": "title", "channel": "My Channel", "title": "Why Is the Sky Blue?"},
  {"type": "statement", "text": "It all starts with sunlight and the atmosphere."},
  {"type": "data", "number": "Rayleigh", "label_below": "Blue scatters 4x more than red", "number_color": "blue"},
  {"type": "handwritten", "text": "The shorter the wavelength,", "emphasis": "the more it scatters."},
  {"type": "end", "text": "Thanks for watching!", "cta": "Subscribe for more science"}
]

# === TIMING (durasi per slide dalam detik) ===
slide_duration = 5.0  # detik per slide

# === JANGAN DIUBAH ===
os.makedirs('patches/patch_01', exist_ok=True)
with open('patches/patch_01/plan_slides.json', 'w') as f:
    json.dump(slides, f, indent=2)
print(f'✅ {len(slides)} slide created')

# Generate timing plan
timing = [{"slide": i+1, "dur": slide_duration} for i in range(len(slides))]
with open('patches/patch_01/plan.json', 'w') as f:
    json.dump(timing, f, indent=2)
total_dur = len(slides) * slide_duration
print(f'✅ Timing plan: {len(slides)} slide x {slide_duration}s = {total_dur:.0f} detik ({total_dur/60:.1f} menit)')

## 3. Generate Voiceover (TTS)
Edit teks narasi sesuai slide di atas.

In [ ]:
# === EDIT NARASI DI SINI ===
narasi = (
    "Why is the sky blue? It all starts with sunlight and the atmosphere. "
    "Rayleigh scattering is the key — blue light scatters four times more than red light. "
    "The shorter the wavelength, the more it scatters. "
    "Thanks for watching, and subscribe for more science!"
)

# === PILIH SUARA ===
# Daftar suara: en-US-ChristopherNeural, en-US-JennyNeural, en-GB-RyanNeural, dll
voice = "en-US-ChristopherNeural"

# === GENERATE ===
!edge-tts --voice {voice} --text "{narasi}" --write-media patches/patch_01/audio.mp3

# Cek durasi
from moviepy import AudioFileClip
clip = AudioFileClip('patches/patch_01/audio.mp3')
print(f'✅ Audio: {clip.duration:.1f} detik')
clip.close()

## 4. Render Video
Proses: HTML slides → PNG → Video MP4

In [ ]:
# Step 1: HTML slides
!python core/slide_builder.py patches/patch_01/plan_slides.json --output patches/patch_01/slides.html
print('✅ HTML slides generated')

In [ ]:
# Step 2: Export PNG (Playwright)
!python core/export_slides.py patches/patch_01
print('✅ PNG exported')

In [ ]:
# Step 3: Build video
# GPU acceleration: Colad punya T4, ffmpeg otomatis pake NVENC
!python video/build_video.py patches/patch_01 --fps 10 --bitrate 2000k --no-music

# Cek hasil
import os
size = os.path.getsize('patches/patch_01/video.mp4') / (1024*1024)
print(f'\n✅ Video selesai! {size:.1f} MB')
print(f'Lokasi: patches/patch_01/video.mp4')

## 5. Download Video
Klik file → Download, atau jalankan kode di bawah untuk download otomatis.

In [ ]:
from google.colab import files
files.download('patches/patch_01/video.mp4')

## 6. Upload ke YouTube (Opsional)
Butuh client_secret.json + refresh_token. Upload langsung dari Colab ke YouTube.

In [ ]:
# Upload file client_secret.json dulu
from google.colab import files
print("Upload client_secret.json:")
uploaded = files.upload()

import base64, json
client_b64 = base64.b64encode(open('client_secret.json', 'rb').read()).decode()
print('\nClient secret dienkripsi:')
print(client_b64)

In [ ]:
# Autentikasi (browser akan terbuka)
from google_auth_oauthlib.flow import InstalledAppFlow

flow = InstalledAppFlow.from_client_secrets_file(
    'client_secret.json',
    scopes=['https://www.googleapis.com/auth/youtube.upload']
)
creds = flow.run_local_server(port=8080)

print(f'\nRefresh Token: {creds.refresh_token}')

# Simpan token
token_data = {
    "token": creds.token,
    "refresh_token": creds.refresh_token,
    "token_uri": creds.token_uri,
    "client_id": creds.client_id,
    "client_secret": creds.client_secret,
    "scopes": creds.scopes,
}
with open('token.json', 'w') as f:
    json.dump(token_data, f)
token_b64 = base64.b64encode(json.dumps(token_data).encode()).decode()
print(f'\nToken B64: {token_b64}')

In [ ]:
# === Upload ke YouTube ===
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth.transport.requests import Request

if creds.expired:
    creds.refresh(Request())

youtube = build('youtube', 'v3', credentials=creds)

body = {
    'snippet': {
        'title': 'Why Is the Sky Blue?',
        'description': 'Learn the science behind the blue sky.\n\n#science #education',
        'tags': ['science', 'education', 'sky'],
        'categoryId': '27'
    },
    'status': {
        'privacyStatus': 'public'
    }
}

media = MediaFileUpload('patches/patch_01/video.mp4', chunksize=-1, resumable=True)

print('Uploading...')
request = youtube.videos().insert(
    part='snippet,status',
    body=body,
    media_body=media
)
response = request.execute()
print(f'✅ Uploaded! https://youtu.be/{response["id"]}')

## Tips
- **Rendering lebih cepat** → Colab GPU (T4) otomatis dipakai ffmpeg
- **Video panjang** → Turunkan FPS ke 5, bitrate 1000k
- **Simpan progress** → File > Save a copy in Drive
- **Background music** → Upload file MP3 ke folder `brand/audio/`